# Batch Size 64 Experiments — Google Colab

This notebook contains the training runs for batch size 64 experiments comparing
Baseline, Masked InfoNCE, and Soft Target InfoNCE loss functions on the IU X-Ray dataset.
It was run on an NVIDIA A100-SXM4-40GB GPU via Google Colab Pro.

---

## How to Run This Notebook in Colab

### Prerequisites

Before opening this notebook, make sure you have the following in your Google Drive:

1. **`medclip-iu.zip`** — the zipped project folder (excluding `.venv/`, `data/images/`, and `.pt` checkpoints)
2. **`NLMCXR_png/`** — the IU X-Ray image folder (~1GB, downloaded from OpenI NLM)

Both should be placed at the same location in your Drive, for example:
```
MyDrive/Colab Notebooks/DeepLearning/Final_Project/medclip-iu.zip
MyDrive/Colab Notebooks/DeepLearning/Final_Project/NLMCXR_png/
```

To create `medclip-iu.zip` from your local machine, run from the parent folder:
```bash
zip -r medclip-iu.zip medclip-iu/ \
  --exclude "medclip-iu/.venv/*" \
  --exclude "medclip-iu/data/images/*" \
  --exclude "medclip-iu/results/*.pt" \
  --exclude "medclip-iu/.git/*" \
  --exclude "medclip-iu/.DS_Store"
```

---

### Step 1 — Set Runtime to GPU

In Colab: `Runtime → Change runtime type → A100 GPU`

---

### Step 2 — Update the base path

At the top of the setup cell, update `BASE` to match where your files live in Drive:

```python
BASE = '/content/drive/MyDrive/Colab Notebooks/DeepLearning/Final_Project'
```

---

### Step 3 — Run cells in order

---

### Note on image copying

Images are copied from Drive to Colab's local `/content/` disk before training.
This is intentional, because I found that reading images directly from Drive over the network was significantly slower and would bottleneck the A100 GPU. The copy takes ~5 minutes but reduces per-batch training time from ~20s to ~1s.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

BASE = '/content/drive/MyDrive/Colab Notebooks/DeepLearning/Final_Project'

with zipfile.ZipFile(f'{BASE}/medclip-iu.zip', 'r') as z:
    z.extractall('/content/')

os.chdir('/content/medclip-iu')

os.makedirs('data', exist_ok=True)
os.symlink(f'{BASE}/NLMCXR_png', 'data/images')

print("Done! Files:", os.listdir('.'))
print("Data:", os.listdir('data'))

Mounted at /content/drive
Done! Files: ['src', 'data', 'results', 'notebooks', '.gitignore', 'requirements.txt']
Data: ['reports', 'images', 'dataset.csv', '.DS_Store']


In [2]:
!pip install transformers pillow pandas scikit-learn tqdm -q

In [6]:
import os, shutil

os.unlink('/content/medclip-iu/data/images')
shutil.copytree(
    '/content/drive/MyDrive/Colab Notebooks/DeepLearning/Final_Project/NLMCXR_png',
    '/content/medclip-iu/data/images'
)
print("Done! Image count:", len(os.listdir('/content/medclip-iu/data/images')))

Done! Image count: 7471


In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-40GB


In [8]:
import sys
sys.path.insert(0, 'src')

!python src/train.py --loss baseline --epochs 5 --batch_size 64
!python src/train.py --loss masked  --epochs 5 --batch_size 64
!python src/train.py --loss soft    --epochs 5 --batch_size 64

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Device: cuda  |  Loss: baseline
Train: 3060  Val: 383  Test: 383
Computing similarity matrix...
Loading weights: 100% 398/398 [00:00<00:00, 864.54it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/5 [train]: 100% 48/48 [00:40<00:00,  1.19it/s]
Epoch 1/5 [val]  : 100% 6/6 [00:0

In [9]:
!python src/eval.py --checkpoint results/baseline_bs64_best.pt --out results/baseline_bs64_metrics.json && \
python src/eval.py --checkpoint results/masked_bs64_best.pt   --out results/masked_bs64_metrics.json && \
python src/eval.py --checkpoint results/soft_bs64_best.pt     --out results/soft_bs64_metrics.json

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Train: 3060  Val: 383  Test: 383
Loading weights: 100% 398/398 [00:00<00:00, 937.62it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding: 100% 6/6 [00:04<00:00,  1.37it/s]

Results for: results/baseline_bs64_best.pt
Metric           Value
----------------------
i2t_R@1         

In [10]:
import shutil, os

save_dir = '/content/drive/MyDrive/Colab Notebooks/DeepLearning/Final_Project/results_bs64'
os.makedirs(save_dir, exist_ok=True)

for f in os.listdir('/content/medclip-iu/results'):
    if f.endswith('.json'):
        shutil.copy(f'/content/medclip-iu/results/{f}', f'{save_dir}/{f}')
        print(f"Saved: {f}")

Saved: soft_bs64_metrics.json
Saved: baseline_metrics.json
Saved: soft_metrics.json
Saved: baseline_bs64_metrics.json
Saved: masked_metrics.json
Saved: masked_bs64_metrics.json
